# Fantasy Premier League Data Loader

Connect to the official FPL API, prepare analysis-ready player, team, position, fixture, and history tables, then run focused data checks and scouting summaries.

In [1]:
import time
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd
import requests

BASE_URL = "https://fantasy.premierleague.com/api"

## 1. API Utilities

Shared request handling and bootstrap-data access.

In [2]:
def fetch_json(url: str, timeout: int = 20) -> Optional[Any]:
    """Fetch JSON from the FPL API, returning None when the request fails."""
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        print(f"Request timed out for: {url}")
    except requests.exceptions.HTTPError as exc:
        print(f"HTTP error for {url}: {exc}")
    except requests.exceptions.RequestException as exc:
        print(f"Request failed for {url}: {exc}")
    except ValueError as exc:
        print(f"Invalid JSON returned by {url}: {exc}")
    return None


def load_bootstrap_data() -> Dict[str, Any]:
    """Fetch the static bootstrap payload or raise a clear error."""
    bootstrap = fetch_json(f"{BASE_URL}/bootstrap-static/")
    if bootstrap is None:
        raise RuntimeError("Unable to load FPL bootstrap-static data.")
    return bootstrap

## 2. Core Tables

Build player, team, and position DataFrames from one bootstrap response.

In [3]:
def build_core_dataframes(
    bootstrap: Optional[Dict[str, Any]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build player, team, and position tables from bootstrap data."""
    bootstrap = bootstrap or load_bootstrap_data()
    elements = bootstrap.get("elements", [])
    teams = bootstrap.get("teams", [])
    positions = bootstrap.get("element_types", [])

    team_map = {
        team["id"]: team.get("name")
        for team in teams
        if team.get("id") is not None
    }
    position_map = {
        position["id"]: (
            position.get("singular_name_short")
            or position.get("singular_name")
            or position.get("name")
        )
        for position in positions
        if position.get("id") is not None
    }

    players_df = pd.DataFrame(elements)
    if not players_df.empty:
        players_df["team_id"] = players_df.get("team")
        players_df["team"] = players_df["team_id"].map(team_map)
        players_df["position_id"] = players_df.get("element_type")
        players_df["position"] = players_df["position_id"].map(position_map)
        players_df["price"] = pd.to_numeric(players_df.get("now_cost"), errors="coerce") / 10
        players_df["xg"] = pd.to_numeric(players_df.get("expected_goals"), errors="coerce")
        players_df["xa"] = pd.to_numeric(players_df.get("expected_assists"), errors="coerce")
        players_df["ict_index"] = pd.to_numeric(players_df.get("ict_index"), errors="coerce")
        players_df["minutes"] = pd.to_numeric(players_df.get("minutes"), errors="coerce")
        players_df["ownership_pct"] = pd.to_numeric(
            players_df.get("selected_by_percent"), errors="coerce"
        )
        players_df["form"] = pd.to_numeric(players_df.get("form"), errors="coerce")
        players_df["status"] = players_df.get(
            "status", pd.Series("", index=players_df.index)
        ).fillna("unknown")
        players_df["availability"] = players_df["status"]
        players_df = players_df.rename(
            columns={"id": "player_id", "web_name": "name"}
        )

    player_columns = [
        "player_id", "name", "team_id", "team", "position_id", "position",
        "price", "total_points", "form", "xg", "xa", "ict_index", "minutes",
        "ownership_pct", "status", "availability",
    ]
    players_df = players_df.reindex(columns=player_columns)

    teams_df = pd.DataFrame(teams).rename(
        columns={
            "id": "team_id",
            "name": "team_name",
            "strength_overall_home": "strength_home",
            "strength_overall_away": "strength_away",
            "strength_attack_home": "attack_strength_home",
            "strength_attack_away": "attack_strength_away",
            "strength_defence_home": "defence_strength_home",
            "strength_defence_away": "defence_strength_away",
        }
    )
    teams_df = teams_df.reindex(
        columns=[
            "team_id", "team_name", "short_name", "strength_home", "strength_away",
            "attack_strength_home", "attack_strength_away", "defence_strength_home",
            "defence_strength_away",
        ]
    )

    positions_df = pd.DataFrame(positions).rename(
        columns={
            "id": "position_id",
            "singular_name": "position_name",
            "singular_name_short": "position_short",
        }
    )
    if "position_name" not in positions_df.columns and "name" in positions_df.columns:
        positions_df["position_name"] = positions_df["name"]
    positions_df = positions_df.reindex(
        columns=["position_id", "position_name", "position_short"]
    )

    return players_df, teams_df, positions_df

## 3. Upcoming Fixtures

Create a team-by-team fixture difficulty table. An empty result retains the same columns, so downstream checks remain valid.

In [4]:
FIXTURE_COLUMNS = [
    "event", "team_id", "team_name", "opponent_team_id",
    "opponent_team_name", "is_home", "fdr", "difficulty",
]


def get_current_gameweek(bootstrap: Dict[str, Any]) -> int:
    """Use the marked current event, or the next unfinished event as a fallback."""
    events = bootstrap.get("events", [])
    current = next((event for event in events if event.get("is_current")), None)
    if current is not None:
        return int(current["id"])

    upcoming = [event for event in events if not event.get("finished", False)]
    if upcoming:
        return int(upcoming[0]["id"])
    return 1


def load_fixtures(
    next_gameweeks: int = 5,
    bootstrap: Optional[Dict[str, Any]] = None,
    fixtures_payload: Optional[List[Dict[str, Any]]] = None,
) -> pd.DataFrame:
    """Load upcoming fixtures as one home and one away row per match."""
    if next_gameweeks < 1:
        raise ValueError("next_gameweeks must be at least 1")

    bootstrap = bootstrap or load_bootstrap_data()
    team_map = {
        team["id"]: team.get("name")
        for team in bootstrap.get("teams", [])
        if team.get("id") is not None
    }
    if fixtures_payload is None:
        fixtures_payload = fetch_json(f"{BASE_URL}/fixtures/")
        if fixtures_payload is None:
            raise RuntimeError("Unable to load FPL fixtures data.")

    first_event = get_current_gameweek(bootstrap)
    last_event = first_event + next_gameweeks - 1
    rows: List[Dict[str, Any]] = []

    for fixture in fixtures_payload:
        event = fixture.get("event")
        if (
            event is None
            or fixture.get("finished") is not False
            or not first_event <= int(event) <= last_event
        ):
            continue

        home_id = fixture.get("team_h")
        away_id = fixture.get("team_a")
        rows.extend(
            [
                {
                    "event": event,
                    "team_id": home_id,
                    "team_name": team_map.get(home_id),
                    "opponent_team_id": away_id,
                    "opponent_team_name": team_map.get(away_id),
                    "is_home": True,
                    "fdr": fixture.get("team_h_difficulty"),
                    "difficulty": fixture.get("team_h_difficulty"),
                },
                {
                    "event": event,
                    "team_id": away_id,
                    "team_name": team_map.get(away_id),
                    "opponent_team_id": home_id,
                    "opponent_team_name": team_map.get(home_id),
                    "is_home": False,
                    "fdr": fixture.get("team_a_difficulty"),
                    "difficulty": fixture.get("team_a_difficulty"),
                },
            ]
        )

    return pd.DataFrame(rows, columns=FIXTURE_COLUMNS)

## 4. Player History

Fetch one player's current and past seasons; bulk requests are throttled to reduce API load.

In [5]:
def get_player_history(player_id: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return current-season and historical-season data for one player."""
    payload = fetch_json(f"{BASE_URL}/element-summary/{player_id}/")
    if payload is None:
        return pd.DataFrame(), pd.DataFrame()

    history_df = pd.DataFrame(payload.get("history", []))
    history_past_df = pd.DataFrame(payload.get("history_past", []))
    if not history_df.empty:
        history_df["player_id"] = int(player_id)
    if not history_past_df.empty:
        history_past_df["player_id"] = int(player_id)
    return history_df, history_past_df


def get_multiple_players_history(
    player_ids: Iterable[int], pause_seconds: float = 0.35
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Fetch several players' histories with a pause between API requests."""
    all_history: List[pd.DataFrame] = []
    all_history_past: List[pd.DataFrame] = []

    for index, player_id in enumerate(player_ids):
        if index:
            time.sleep(pause_seconds)
        history_df, history_past_df = get_player_history(int(player_id))
        if not history_df.empty:
            all_history.append(history_df)
        if not history_past_df.empty:
            all_history_past.append(history_past_df)

    combined_history = (
        pd.concat(all_history, ignore_index=True) if all_history else pd.DataFrame()
    )
    combined_history_past = (
        pd.concat(all_history_past, ignore_index=True)
        if all_history_past
        else pd.DataFrame()
    )
    return combined_history, combined_history_past

## 5. Load Data

Run the next cell to fetch fresh API data. It makes one bootstrap request, one fixture request, and one player-history request.

In [6]:
bootstrap_data = load_bootstrap_data()
players_df, teams_df, positions_df = build_core_dataframes(bootstrap_data)
fixtures_df = load_fixtures(next_gameweeks=5, bootstrap=bootstrap_data)

sample_player_id = int(players_df["player_id"].iloc[0]) if not players_df.empty else None
if sample_player_id is not None:
    history_df, history_past_df = get_player_history(sample_player_id)
else:
    history_df, history_past_df = pd.DataFrame(), pd.DataFrame()

frames = {
    "players": players_df,
    "teams": teams_df,
    "positions": positions_df,
    "fixtures": fixtures_df,
    "history": history_df,
}

for frame_name, dataframe in frames.items():
    print(f"{frame_name}: {dataframe.shape}")

print("\nPlayer sample:")
display(players_df.head())
print("\nTeam sample:")
display(teams_df.head())
print("\nPosition table:")
display(positions_df)
print("\nUpcoming fixture sample:")
display(fixtures_df.head())
print("\nPlayer history sample:")
display(history_df.head())

players: (667, 16)
teams: (20, 9)
positions: (4, 3)
fixtures: (80, 8)
history: (5, 42)

Player sample:


,player_id,name,team_id,team,position_id,position,price,total_points,form,xg,xa,ict_index,minutes,ownership_pct,status,availability
0,1,Raya,1,Arsenal,1,GKP,6.1,30,6.0,0.00,0.01,10.6,450,42.3,a,a
1,2,Arrizabalaga,1,Arsenal,1,GKP,4.9,0,0.0,0.00,0.00,0.0,0,0.1,a,a
2,3,Meslier,1,Arsenal,1,GKP,4.9,0,0.0,0.00,0.00,0.0,0,0.0,a,a
3,4,Gabriel,1,Arsenal,2,DEF,8.0,25,5.0,0.51,0.20,20.0,450,22.7,a,a
4,5,J.Timber,1,Arsenal,2,DEF,6.5,1,0.2,0.00,0.03,1.5,104,0.4,a,a



Team sample:


,team_id,team_name,short_name,strength_home,strength_away,attack_strength_home,attack_strength_away,defence_strength_home,defence_strength_away
0,1,Arsenal,ARS,4,5,0,0,0,0
1,2,Aston Villa,AVL,3,3,0,0,0,0
2,3,Bournemouth,BOU,3,3,0,0,0,0
3,4,Brentford,BRE,3,3,0,0,0,0
4,5,Brighton,BHA,3,4,0,0,0,0



Position table:


,position_id,position_name,position_short
0,1,Goalkeeper,GKP
1,2,Defender,DEF
2,3,Midfielder,MID
3,4,Forward,FWD



Upcoming fixture sample:


,event,team_id,team_name,opponent_team_id,opponent_team_name,is_home,fdr,difficulty
0,6,1,Arsenal,13,Leeds,True,3,3
1,6,13,Leeds,1,Arsenal,False,5,5
2,6,2,Aston Villa,4,Brentford,True,3,3
3,6,4,Brentford,2,Aston Villa,False,3,3
4,6,6,Chelsea,3,Bournemouth,True,3,3



Player history sample:


,element,fixture,opponent_team,total_points,was_home,kickoff_time,team_h_score,team_a_score,round,modified,...,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,value,transfers_balance,selected,transfers_in,transfers_out,player_id
0,1,1,7,6,True,2026-08-21T19:00:00Z,3,0,1,False,...,0.00,0.00,0.00,0.20,60,0,3347153,0,0,1
1,1,20,2,6,False,2026-08-31T19:00:00Z,0,1,2,False,...,0.00,0.00,0.00,0.33,60,24756,3801300,106734,81978,1
2,1,29,6,3,True,2026-09-06T15:30:00Z,2,1,3,False,...,0.00,0.01,0.01,0.39,60,-23780,3887669,204948,228728,1
3,1,36,20,14,False,2026-09-12T19:00:00Z,0,2,4,False,...,0.00,0.00,0.00,1.80,60,34449,4195206,249119,214670,1
4,1,42,5,1,False,2026-09-19T14:00:00Z,3,0,5,False,...,0.00,0.00,0.00,1.31,60,212570,4487209,279598,67028,1


## 6. Data Quality Checks

Review table structure, missing values, duplicate keys, and team/position mappings before interpreting the rankings.

In [7]:
print("Missing values:")
for frame_name, dataframe in frames.items():
    if dataframe.empty:
        print(f"{frame_name}: no rows")
        continue
    missing = dataframe.isna().sum()
    missing = missing[missing > 0]
    print(f"{frame_name}: {missing.to_dict() if not missing.empty else 'none'}")

print("\nDuplicate key checks:")
key_checks = {
    "players": (players_df, ["player_id"]),
    "teams": (teams_df, ["team_id"]),
    "positions": (positions_df, ["position_id"]),
    "history": (history_df, ["player_id", "fixture"]),
}
for frame_name, (dataframe, keys) in key_checks.items():
    if dataframe.empty or not set(keys).issubset(dataframe.columns):
        print(f"{frame_name}: not enough data to check")
    else:
        duplicate_count = int(dataframe.duplicated(subset=keys).sum())
        print(f"{frame_name} duplicate {keys}: {duplicate_count}")

print("\nMapping checks:")
for frame_name, dataframe, columns in [
    ("players", players_df, ["team", "position"]),
    ("fixtures", fixtures_df, ["team_name", "opponent_team_name"]),
]:
    if dataframe.empty:
        print(f"{frame_name}: no rows to validate")
    elif not set(columns).issubset(dataframe.columns):
        print(f"{frame_name}: required columns are unavailable")
    else:
        missing_count = int(dataframe[columns].isna().any(axis=1).sum())
        print(f"{frame_name}: {missing_count} rows with missing mappings")

Missing values:
players: none
teams: none
positions: none
fixtures: none
history: none

Duplicate key checks:
players duplicate ['player_id']: 0
teams duplicate ['team_id']: 0
positions duplicate ['position_id']: 0
history duplicate ['player_id', 'fixture']: 0

Mapping checks:
players: 0 rows with missing mappings
fixtures: 0 rows with missing mappings


In [8]:
numeric_checks = {
    "players": ["price", "total_points", "form", "xg", "xa", "ict_index", "minutes", "ownership_pct"],
    "fixtures": ["event", "fdr", "difficulty"],
    "history": ["total_points", "minutes", "goals_scored", "assists", "clean_sheets", "goals_conceded"],
}

for frame_name, columns in numeric_checks.items():
    dataframe = frames[frame_name]
    print(f"\n{frame_name} numeric summary:")
    available = [column for column in columns if column in dataframe.columns]
    if dataframe.empty or not available:
        print("No values available")
        continue
    print(dataframe[available].describe().T[["min", "max", "mean"]])


players numeric summary:
               min     max        mean
price          3.9   15.60    5.086057
total_points  -1.0   47.00    6.953523
form          -0.3   11.30    1.398951
xg             0.0    4.42    0.229490
xa             0.0    1.71    0.138336
ict_index      0.0   54.60    7.769715
minutes        0.0  450.00  147.514243
ownership_pct  0.0   73.80    2.248726

fixtures numeric summary:
            min  max   mean
event       6.0  9.0  7.500
fdr         2.0  5.0  3.075
difficulty  2.0  5.0  3.075

history numeric summary:
                 min   max  mean
total_points     1.0  14.0   6.0
minutes         90.0  90.0  90.0
goals_scored     0.0   0.0   0.0
assists          0.0   0.0   0.0
clean_sheets     0.0   1.0   0.6
goals_conceded   0.0   3.0   0.8


## 7. Exploratory Scouting

Use the cleaned tables to compare positions, rank overall output, find value, and scan upcoming fixtures.

In [18]:
position_summary = (
    players_df.groupby("position", as_index=False)
    .agg(
        players=("player_id", "count"),
        avg_price=("price", "mean"),
        avg_points=("total_points", "mean"),
        avg_xg=("xg", "mean"),
        avg_xa=("xa", "mean"),
        avg_ict=("ict_index", "mean"),
    )
    .sort_values("avg_points", ascending=False)
)
display(position_summary)

,position,players,avg_price,avg_points,avg_xg,avg_xa,avg_ict
0,DEF,217,4.628111,7.751152,0.129401,0.117005,7.629493
3,MID,298,5.403356,7.016779,0.260134,0.201544,8.886242
1,FWD,79,5.694937,6.303797,0.600886,0.082152,8.034177
2,GKP,73,4.493151,5.027397,0.000000,0.004521,3.342466


In [10]:
player_rankings = players_df.sort_values(
    ["total_points", "xg", "xa"], ascending=False
).head(15)
display(
    player_rankings[
        ["name", "position", "team", "price", "total_points", "form", "xg", "xa", "ownership_pct"]
    ]
)

,name,position,team,price,total_points,form,xg,xa,ownership_pct
136,Groß,MID,Brighton,5.8,47,10.7,1.77,0.86,28.8
272,Tarkowski,DEF,Everton,6.1,43,8.3,0.18,0.04,16.4
403,Bogle,DEF,Leeds,4.6,42,11.3,1.38,0.08,7.2
491,Haaland,FWD,Man City,15.6,39,8.0,4.42,0.53,73.8
105,Schade,MID,Brentford,6.2,39,8.7,1.52,0.32,11.5
127,De Cuyper,DEF,Brighton,5.0,38,7.0,1.55,0.67,27.7
474,Gvardiol,DEF,Man City,5.7,37,7.7,0.22,0.74,26.8
482,Cherki,MID,Man City,7.8,34,4.0,0.72,1.54,28.4
351,Tzolakis,GKP,Hull City,4.7,34,4.7,0.00,0.00,12.8
459,Isak,FWD,Liverpool,9.1,33,7.7,3.54,0.08,22.4


In [11]:
players_df["points_per_price"] = (
    players_df["total_points"] / players_df["price"].replace(0, pd.NA)
)
value_players = players_df.sort_values(
    ["points_per_price", "total_points"], ascending=False
).head(15)
display(
    value_players[
        ["name", "position", "team", "price", "total_points", "points_per_price", "ownership_pct"]
    ]
)

,name,position,team,price,total_points,points_per_price,ownership_pct
403,Bogle,DEF,Leeds,4.6,42,9.130435,7.2
136,Groß,MID,Brighton,5.8,47,8.103448,28.8
127,De Cuyper,DEF,Brighton,5.0,38,7.600000,27.7
351,Tzolakis,GKP,Hull City,4.7,34,7.234043,12.8
272,Tarkowski,DEF,Everton,6.1,43,7.049180,16.4
474,Gvardiol,DEF,Man City,5.7,37,6.491228,26.8
327,Ajayi,DEF,Hull City,4.2,27,6.428571,14.0
151,Vuskovic,DEF,Brighton,5.0,32,6.400000,3.6
325,Egan,DEF,Hull City,4.1,26,6.341463,7.0
105,Schade,MID,Brentford,6.2,39,6.290323,11.5


In [12]:
fixtures_by_team = (
    fixtures_df.groupby(["team_name", "event"], as_index=False)[["difficulty", "is_home"]]
    .mean()
    .sort_values(["team_name", "event"])
)
display(fixtures_by_team.head(20))

,team_name,event,difficulty,is_home
0,Arsenal,6,3.0,1.0
1,Arsenal,7,3.0,0.0
2,Arsenal,8,3.0,1.0
3,Arsenal,9,4.0,0.0
4,Aston Villa,6,3.0,1.0
5,Aston Villa,7,3.0,0.0
6,Aston Villa,8,4.0,1.0
7,Aston Villa,9,2.0,1.0
8,Bournemouth,6,4.0,0.0
9,Bournemouth,7,3.0,1.0


In [13]:
shortlist = players_df.loc[
    (players_df["minutes"] >= 500)
    & (players_df["total_points"] >= 40)
    & (players_df["price"] <= 7.5)
].sort_values(["total_points", "xg", "ownership_pct"], ascending=False)
display(
    shortlist[
        ["name", "position", "team", "price", "total_points", "xg", "xa", "minutes", "ownership_pct"]
    ].head(20)
)

,name,position,team,price,total_points,xg,xa,minutes,ownership_pct


## 8. Offline Smoke Test

Check the table transformations and empty-fixture handling using sample data, without making API requests.

In [14]:
sample_bootstrap = {
    "events": [{"id": 4, "is_current": True, "finished": False}],
    "teams": [
        {"id": 1, "name": "Arsenal"},
        {"id": 2, "name": "Aston Villa"},
    ],
    "element_types": [
        {"id": 1, "singular_name": "Goalkeeper", "singular_name_short": "GKP"},
    ],
    "elements": [
        {
            "id": 10,
            "web_name": "Sample Keeper",
            "team": 1,
            "element_type": 1,
            "now_cost": 50,
            "total_points": 12,
            "form": "4.0",
            "expected_goals": "0.0",
            "expected_assists": "0.1",
            "ict_index": "3.0",
            "minutes": 180,
            "selected_by_percent": "2.5",
            "status": "a",
        },
    ],
}

sample_players, _, _ = build_core_dataframes(sample_bootstrap)
assert sample_players.loc[0, "team"] == "Arsenal"
assert sample_players.loc[0, "position"] == "GKP"
assert sample_players.loc[0, "form"] == 4.0

empty_fixtures = load_fixtures(
    bootstrap=sample_bootstrap,
    fixtures_payload=[],
)
assert empty_fixtures.empty
assert list(empty_fixtures.columns) == FIXTURE_COLUMNS

sample_fixtures = load_fixtures(
    bootstrap=sample_bootstrap,
    fixtures_payload=[
        {
            "event": 4,
            "finished": False,
            "team_h": 1,
            "team_a": 2,
            "team_h_difficulty": 2,
            "team_a_difficulty": 4,
        }
    ],
)
assert len(sample_fixtures) == 2
assert sample_fixtures.loc[0, "team_name"] == "Arsenal"
assert sample_fixtures.loc[1, "opponent_team_name"] == "Arsenal"
print("Offline smoke test passed.")

Offline smoke test passed.
